In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col,split,explode,size
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

spark = (
    SparkSession.builder
    .appName("iceberg-learning")
    .master("local[*]")
    .config(
        "spark.jars.packages",
        "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.5.2"
    )
    .config(
        "spark.sql.extensions",
        "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions"
    )
    .config(
        "spark.sql.catalog.local",
        "org.apache.iceberg.spark.SparkCatalog"
    )
    .config(
        "spark.sql.catalog.local.type",
        "hadoop"
    )
    .config(
        "spark.sql.catalog.local.warehouse",
        "/data/warehouse"
    )
    .getOrCreate()
)

df = spark.read.parquet("/data/raw/machine-raw-log.snappy.parquet").select("ChassisId","Cell","LogId","CellValue","CreationDateTime")

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/jmattam/.ivy2/cache
The jars for the packages stored in: /home/jmattam/.ivy2/jars
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-ea086feb-9801-4802-9f12-f0901b9aee7e;1.0
	confs: [default]
	found org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.5.2 in central
downloading https://repo1.maven.org/maven2/org/apache/iceberg/iceberg-spark-runtime-3.5_2.12/1.5.2/iceberg-spark-runtime-3.5_2.12-1.5.2.jar ...
	[SUCCESSFUL ] org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.5.2!iceberg-spark-runtime-3.5_2.12.jar (2384ms)
:: resolution report :: resolve 936ms :: artifacts dl 2389ms
	:: modules in use:
	org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.5.2 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evict

In [2]:
df.columns

['ChassisId', 'Cell', 'LogId', 'CellValue', 'CreationDateTime']

In [3]:
mapping_schema = StructType([
    StructField("PIPELINE", StringType(), True),
    StructField("LOG_ID", IntegerType(), True),
    StructField("Cell", StringType(), True),
    StructField("MEASURE_CODE_ACC", StringType(), True),
    StructField("UNIT", DoubleType(), True),
    StructField("TARGET_UNIT", StringType(), True),
    StructField("DESCRIPTION", StringType(), True)
])

df_mapping = spark.read.option("header", "true").schema(mapping_schema).csv("/data/raw/log_cell_mapping.csv")

In [4]:
df_mapping = df_mapping.withColumn("Cell_Array",split(df_mapping['Cell'],",")).withColumn("Cell_number",explode("Cell_Array")).withColumn("Cell_Count",size(col("Cell_Array")))

In [5]:
df_mapping.show()

+------------+------+----+--------------------+-----+-----------+--------------------+----------+-----------+----------+
|    PIPELINE|LOG_ID|Cell|    MEASURE_CODE_ACC| UNIT|TARGET_UNIT|         DESCRIPTION|Cell_Array|Cell_number|Cell_Count|
+------------+------+----+--------------------+-----+-----------+--------------------+----------+-----------+----------+
|      HAULER|  1061|   1|     ENG_HOURS_TOTAL| 0.05|      Hours|Total cumulative ...|       [1]|          1|         1|
|      HAULER|  1061| 1,2|     TOTAL_FUEL_USED|0.001|     Liters|Cumulative engine...|    [1, 2]|          1|         2|
|      HAULER|  1061| 1,2|     TOTAL_FUEL_USED|0.001|     Liters|Cumulative engine...|    [1, 2]|          2|         2|
|      HAULER|  1080|   3|    IDLE_HOURS_TOTAL|  0.1|      Hours|Total engine idle...|       [3]|          3|         1|
|      HAULER|  1080| 4,5|      IDLE_FUEL_USED|0.001|     Liters|Total fuel consum...|    [4, 5]|          4|         2|
|      HAULER|  1080| 4,5|      

In [6]:
lkpDF = df_mapping.select("MEASURE_CODE_ACC","LOG_ID","Cell_number","UNIT","Cell_Array","Cell_Count","PIPELINE")
cond = [((df.LogId == lkpDF.LOG_ID) &\
         (df.Cell == lkpDF.Cell_number))]
mergedDF = df.join(lkpDF,on = cond, how = "inner" )
mergedDF = mergedDF.withColumn("CellValue_new",col("CellValue")*col("UNIT"))

In [7]:
mergedDF.show()

+-----------+----+-----+---------+--------------------+----------------+------+-----------+-----+----------+----------+--------+------------------+
|  ChassisId|Cell|LogId|CellValue|    CreationDateTime|MEASURE_CODE_ACC|LOG_ID|Cell_number| UNIT|Cell_Array|Cell_Count|PIPELINE|     CellValue_new|
+-----------+----+-----+---------+--------------------+----------------+------+-----------+-----+----------+----------+--------+------------------+
|H100E121013|   1| 1061|  2083007|2021-05-24 15:33:...| TOTAL_FUEL_USED|  1061|          1|0.001|    [1, 2]|         2|  HAULER|          2083.007|
|H100E121013|   1| 1061|  2083007|2021-05-24 15:33:...| ENG_HOURS_TOTAL|  1061|          1| 0.05|       [1]|         1|  HAULER|         104150.35|
|H100E121013|   2| 1061|      577|2021-05-24 15:33:...| TOTAL_FUEL_USED|  1061|          2|0.001|    [1, 2]|         2|  HAULER|             0.577|
|H100E121015|   1| 1061|  2336943|2021-05-24 15:43:...| TOTAL_FUEL_USED|  1061|          1|0.001|    [1, 2]|    

In [8]:
groupbyCols = [
    "ChassisId",
    "CreationDateTime",
    "LogId",
    "MEASURE_CODE_ACC",
    "PIPELINE"
]
mergedDF = mergedDF.groupBy(groupbyCols).agg(F.sum("CellValue_new").alias("MEASURE_CODE_ACC_VALUE"))

In [9]:
mergedDF.show()

[Stage 5:>                                                          (0 + 1) / 1]

+-----------+--------------------+-----+----------------+--------+----------------------+
|  ChassisId|    CreationDateTime|LogId|MEASURE_CODE_ACC|PIPELINE|MEASURE_CODE_ACC_VALUE|
+-----------+--------------------+-----+----------------+--------+----------------------+
|H100E121013|2021-05-25 12:33:...| 1061| ENG_HOURS_TOTAL|  HAULER|             107930.35|
|H100E121015|2021-05-30 19:07:...| 1061| TOTAL_FUEL_USED|  HAULER|              2802.326|
|H100E121012|2021-05-29 23:51:...| 1061| ENG_HOURS_TOTAL|  HAULER|              109910.8|
|H100E121013|2021-05-28 01:33:...| 1061| TOTAL_FUEL_USED|  HAULER|    2347.0910000000003|
|H100E121012|2021-05-28 11:21:...| 1061| ENG_HOURS_TOTAL|  HAULER|              107522.6|
|H100E121015|2021-05-24 23:43:...| 1061| TOTAL_FUEL_USED|  HAULER|              2366.595|
|H100E121012|2021-05-31 01:21:...| 1061| TOTAL_FUEL_USED|  HAULER|              2287.869|
|H100E121013|2021-05-30 18:03:...| 1061| ENG_HOURS_TOTAL|  HAULER|    127135.45000000001|
|H100E1210

In [10]:
mergedDF.writeTo("local.db.merged_logs").create()

In [11]:
spark.sql("USE local.db")
spark.sql("SELECT * FROM merged_logs LIMIT 5").show()

+-----------+--------------------+-----+----------------+--------+----------------------+
|  ChassisId|    CreationDateTime|LogId|MEASURE_CODE_ACC|PIPELINE|MEASURE_CODE_ACC_VALUE|
+-----------+--------------------+-----+----------------+--------+----------------------+
|H100E121013|2021-05-25 12:33:...| 1061| ENG_HOURS_TOTAL|  HAULER|             107930.35|
|H100E121015|2021-05-30 19:07:...| 1061| TOTAL_FUEL_USED|  HAULER|              2802.326|
|H100E121012|2021-05-29 23:51:...| 1061| ENG_HOURS_TOTAL|  HAULER|              109910.8|
|H100E121013|2021-05-28 01:33:...| 1061| TOTAL_FUEL_USED|  HAULER|    2347.0910000000003|
|H100E121012|2021-05-28 11:21:...| 1061| ENG_HOURS_TOTAL|  HAULER|              107522.6|
+-----------+--------------------+-----+----------------+--------+----------------------+



In [12]:
spark.sql("""
    UPDATE local.db.merged_logs
    SET MEASURE_CODE_ACC_VALUE = 999.0
    WHERE ChassisId = 'H100E121013'
    AND MEASURE_CODE_ACC = 'ENG_HOURS_TOTAL'
""")

DataFrame[]

In [13]:
spark.sql("""
    DELETE FROM local.db.merged_logs
    WHERE LogId = 1061
    AND ChassisID = 'H100E121013'
""")

DataFrame[]

In [14]:
spark.sql("SELECT * FROM local.db.merged_logs.snapshots").show(truncate=False)

+-----------------------+-------------------+-------------------+---------+------------------------------------------------------------------------------------------------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|committed_at           |snapshot_id        |parent_id          |operation|manifest_list                                                                                               |summary                                                                                                                                                                                                                                                        

In [15]:
spark.sql("SHOW TBLPROPERTIES local.db.merged_logs").show(truncate=False)

+-------------------------------+------------------+
|key                            |value             |
+-------------------------------+------------------+
|current-snapshot-id            |931095857651599677|
|format                         |iceberg/parquet   |
|format-version                 |2                 |
|write.parquet.compression-codec|zstd              |
+-------------------------------+------------------+

